# 05 · The end-to-end pipeline

**What it does.** Runs `domains/vrp/run.py` exactly as `python -m domains.vrp.run --config configs/vrp.yaml`,
once per solver: build and brute-force-check the Stage 1 QUBO → solve → decode the lowest-energy feasible
sample into customer groups → build, check and solve one Stage 2 QUBO per group → combine into routes.
Every solver run is saved as a JSON record.

**Why it was implemented.** The brief asks for a reproducible implementation with documented parameters and
results. One entry point with one configuration file (all settings and seeds) gives that, and the JSON
records carry every field the project conventions require (`CLAUDE.md`, "Every run saves a JSON result").
This notebook writes its records to a temporary folder so the committed `results/` stay unchanged.

**Resources.**
- Code: `domains/vrp/run.py` (`run`, `solve_stage`), `configs/vrp.yaml`, everything from notebooks 01–04.
- Libraries: PyYAML, plus the solver libraries.
- Stored outputs of the reported run: `results/vrp_*_20261018T0956*.json`.

In [ ]:
# Run from anywhere: move to the repository root so `common/`, `domains/`, `data/` resolve.
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repository root:", ROOT)

In [ ]:
import tempfile
import yaml
from domains.vrp.run import run

cfg = yaml.safe_load(Path("configs/vrp.yaml").read_text(encoding="utf-8"))
cfg["results_dir"] = tempfile.mkdtemp(prefix="vrp_notebook_")
print("solvers:", cfg["solvers"], "| QAOA:", {k: cfg["qaoa"][k] for k in ("reps", "optimizer", "maxiter", "shots")})
summaries = run(cfg)  # under a minute
for s in summaries:
    print(f"{s['solver']:<4} groups {s['clusters']} -> routes {s['routes']} | cost {s['objective']:g} | feasible {s['feasible']}")

## What each run stores
Each stage record holds the full measured distribution, the formulation check, statistics, settings, seeds,
penalties and package versions; the chain summary holds the routes and the objective.

In [ ]:
from common.results import load_result

qaoa_summary = next(s for s in summaries if s["solver"] == "qaoa")
stage1 = load_result(qaoa_summary["stage1_result"])
required = ["domain", "instance", "n_vars", "solver", "backend", "reps", "optimizer", "maxiter", "initial_gammas",
            "initial_betas", "shots", "seeds", "penalties", "best_feasible", "stats", "runtime_s",
            "package_versions", "distribution", "formulation_check"]
missing = [k for k in required if k not in stage1]
print("record keys:", sorted(stage1))
print("missing required keys:", missing or "none")
print("stats:", {k: stage1["stats"][k] for k in ("p_optimal", "p_feasible", "total_samples", "n_distinct")})
for path in qaoa_summary["stage2_results"]:
    r = load_result(path)
    print(f"stage 2 route {r['route']}: {r['n_vars']} qubits, p_optimal {r['stats']['p_optimal']:.3f}, "
          f"p_feasible {r['stats']['p_feasible']:.3f}")

## Expected results
- Both solvers: groups [[1, 2, 5], [3, 4]], routes 0-1-5-2-0 and 0-4-3-0 (QAOA may return a reversed tour), cost **118**, feasible.
- QAOA records: Stage 1 p_optimal ≈ 0.052; route {1, 2, 5} ≈ 0.177; route {3, 4} ≈ 0.828.
- No required field missing.

Remember the caveat from the report: the chain keeps the best of 4096 shots, which at 256–512 bitstrings is
close to enumeration, so the 118 says little about QAOA; p_optimal and p_feasible do.

In [ ]:
assert all(s["objective"] == 118 and s["feasible"] and s["clusters"] == [[1, 2, 5], [3, 4]] for s in summaries)
assert not missing
assert abs(stage1["stats"]["p_optimal"] - 0.052) < 0.005